# Lumina Desk — Advanced 'Hey Lumina' Wake Word Training

Higher-quality version of openWakeWord's automatic trainer, tuned for the phrase **hey lumina**. Everything is synthetic — you do **not** record your voice.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all. Expect roughly **60-120 min** (bigger data = better model).


## 1. Environment setup

In [ ]:
# install piper-sample-generator (synthetic TTS) and openWakeWord training deps
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

import os
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


In [ ]:
import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


## 2. Download data (RIRs, background noise, music, features)

In [ ]:
# Room impulse responses (make the wake word robust to room echo)
output_dir = './mit_rirs'
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


In [ ]:
# ADVANCED: more background noise (several AudioSet parts) + more music (FMA)
os.makedirs('audioset', exist_ok=True)

# Several balanced-train parts for diverse noise (restaurant-like chatter, clatter, music)
audioset_parts = ['bal_train06.tar', 'bal_train07.tar', 'bal_train08.tar', 'bal_train09.tar']
for fname in audioset_parts:
    out = f'audioset/{fname}'
    link = 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/' + fname
    !wget -O {out} {link}
    !cd audioset && tar -xf {fname}

output_dir = './audioset_16k'
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive — more hours than the quick example (music is a common false-trigger source)
output_dir = './fma'
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
n_hours = 4
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    if i == n_hours*3600//30 - 1:
        break


In [ ]:
# Pre-computed openWakeWord features: ~2,000 hrs negatives + validation set
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy


## 3. Training configuration (advanced 'hey lumina')

Bigger than the quick example: **30,000** positive + negative synthetic samples, **3,000** validation, **50,000** training steps, and production target metrics (accuracy >= 0.7, recall >= 0.5, false positives <= 0.2/hr). openWakeWord also auto-generates *adversarial* phrases (words that sound like 'hey lumina') to cut false triggers.

In [ ]:
config = yaml.load(open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader)
config


In [ ]:
# ---- Advanced 'hey lumina' settings ----
config['target_phrase'] = ['hey lumina']
config['model_name'] = 'hey_lumina'

config['n_samples'] = 20000        # synthetic positives (plenty; finishes faster)
config['n_samples_val'] = 2000     # validation examples for early stopping
config['steps'] = 30000            # strong training, but likely to finish on free Colab

# Production target metrics (higher quality than the quick-example defaults)
config['target_accuracy'] = 0.7
config['target_recall'] = 0.5
config['target_false_positive_rate'] = 0.2

config['background_paths'] = ['./audioset_16k', './fma']
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print('config written for', config['target_phrase'], '->', config['model_name'])


## 4. Train

In [ ]:
# Step 1: generate synthetic 'hey lumina' clips (and adversarial negatives)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips


In [ ]:
# Step 2: augment clips with room echo + background noise
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips


In [ ]:
# Step 3: train the model (this is the long step)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model


In [ ]:
# Step 4 (optional): re-save tflite if Colab didn't (we only need the .onnx anyway)
def convert_onnx_to_tflite(onnx_model_path, output_path):
    import onnx, logging, tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device='CPU')
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, 'tf_model'))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, 'tf_model'))
        tflite_model = converter.convert()
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

try:
    convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")
except Exception as e:
    print('tflite convert skipped:', e)


## 5. Save the model (to Google Drive AND direct download)

IMPORTANT: Colab wipes its temporary storage when the runtime disconnects, so we save the model to **Google Drive** first — that copy survives even if the runtime recycles. It appears in your Drive as `hey_lumina.onnx`.

In [ ]:
# Save the trained model to Google Drive so it can't be lost to a runtime recycle
from google.colab import drive
drive.mount('/content/drive')

import shutil
src = f"my_custom_model/{config['model_name']}.onnx"
dst = '/content/drive/MyDrive/hey_lumina.onnx'
shutil.copy(src, dst)
print('Saved to Google Drive:', dst)
print('Size:', os.path.getsize(dst), 'bytes')


### Get it onto the Pi
- The model is now in your **Google Drive** as `hey_lumina.onnx` (open drive.google.com).
- Download it to the Pi and run:
```bash
cp ~/Downloads/hey_lumina.onnx ~/lumina-desk/models/
```
Lumina Desk auto-detects it and the wake word becomes **Hey Lumina**.

In [ ]:
# Also trigger a direct browser download (backup to the Drive copy)
from google.colab import files
files.download(f"my_custom_model/{config['model_name']}.onnx")
